# GN model 3편 + HCF geometry: 독립 검증 — final-2final3-final4

이번 버전은 기존 GN→HCF geometry 흐름을 보존하면서 **정량 출력이 공개된 네 연구 항목**을 분리한다.

1. **Poggiolini (JLT 2012)**: Eq.13과 기존 closed-form eta의 수식 구현 일관성 검산.
2. **Nespola et al. (PTL 2014)**: 7종 광섬유의 PM-16QAM 실험 Lmax와 현재 GN 코드의 예측 Lmax 비교. 논문 Table I의 입력값을 그대로 사용하며 실험 Lmax는 피팅에 사용하지 않는다.
3. **Carena et al. (JLT 2012)**: PM-BPSK/PM-QPSK/PM-8QAM/PM-16QAM, 32 GBaud, 50 GHz→symbol-rate 간격 검증 논문. 첨부 그림의 원본 캡션·패널별 전체 입력값이 확인되지 않아 자동 오차를 산출하지 않는다. 조건 CSV를 넣으면 실행되는 검증 인터페이스를 제공한다.
4. **Petrovich et al. (Nature Photonics 2025)**: HCF2 FEM 분산과 측정 손실을 Raw Hasan/MS 및 Raw bouncing-ray 결과와 비교한다. 분산은 모델 대 FEM, 손실은 proxy 대 총손실의 모델 층 진단이다.

`reference output`은 계산이 끝난 뒤 오차를 구하는 데만 쓰며 $f_1,f_2$, GN 계수, 손실 계수에 피팅하지 않는다. Nespola의 실험은 `Lmax` 및 fiber parameter가 표로 공개되어 있으므로 기존 첨부 그림과 달리 동일 입력 정량검증이 가능하다.

**주요 출처:** [Poggiolini GN 저자 원문](https://iris.polito.it/handle/11583/2506445), [Nespola 공개 PDF](https://backoffice.biblio.ugent.be/download/5743918/5743952), [Carena 논문 초록](https://opg.optica.org/abstract.cfm?uri=jlt-30-10-1524), [Petrovich preprint](https://arxiv.org/abs/2503.21467).


In [ ]:
import json, math, pathlib, shutil, platform, sys
from dataclasses import dataclass, replace, asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar, brentq
from scipy.special import erfc, erfcinv
from numpy.polynomial.hermite import hermgauss
try:
    from IPython.display import display
except ImportError:
    def display(obj): print(obj.to_string(index=False) if isinstance(obj,pd.DataFrame) else obj)

RESULTS = pathlib.Path('hcf_results')
RESULTS.mkdir(exist_ok=True)
TABLES = {}
FIGURES = []
CHECKS = []
def table(name, frame, show=True):
    TABLES[name] = frame.copy()
    frame.to_csv(RESULTS/(name+'.csv'), index=False)
    if show: display(frame)
def figure(name, fig):
    fig.savefig(RESULTS/(name+'.png'), dpi=170, bbox_inches='tight')
    fig.savefig(RESULTS/(name+'.svg'), bbox_inches='tight')
    FIGURES.append(name)
    plt.show()
def check(name, condition):
    assert bool(condition), name
    CHECKS.append({'check':name,'status':'PASS'})
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':170,'axes.grid':True,'grid.alpha':.2,
                     'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
pd.set_option('display.max_columns',30)
pd.set_option('display.width',180)
C0 = 299792458.0
H_PLANCK = 6.62607015e-34
U01 = 2.4048255577
DB_PER_NEPER = 10/np.log(10)
def db2lin(x): return 10**(np.asarray(x,dtype=float)/10)
def lin2db(x): return 10*np.log10(np.asarray(x,dtype=float))
def dbm2w(x): return 1e-3*db2lin(x)
def w2dbm(x): return lin2db(x)+30
def signed_error_percent(model, reference):
    model, reference = np.broadcast_arrays(np.asarray(model,dtype=float),np.asarray(reference,dtype=float))
    return np.divide(100*(model-reference),reference,out=np.full_like(model,np.nan),where=reference!=0)
def error_metrics(model, reference):
    delta=np.asarray(model)-np.asarray(reference)
    return {'MAPE_percent':float(np.nanmean(np.abs(signed_error_percent(model,reference)))),
            'RMSE':float(np.sqrt(np.mean(delta**2))), 'MAE':float(np.mean(np.abs(delta)))}
print('No output fitting. Scenario assumptions are separate from reference observations.')


## 1. HCF geometry → 손실 오차

HCF2 입력: 코어 반경 14.75 μm, 대표 막 두께 0.50 μm, 외부/중간/내부 튜브 직경 31.05/23.75/7.70 μm, 외부 튜브 5개, 이중 중첩. 기하 치수의 실제 범위는 Petrovich 원문을 참조한다. 대표 막 두께는 기존 코드 입력을 유지한다.

아래 정의는 기존 노트북의 함수를 복사한 것이다. **손실 함수는 코어 반경과 대표 막 두께만 사용한다. 중첩 튜브 직경·튜브 수·간격을 쓰지 않으므로 DNANF 전체 구조의 총손실 모델이 아니다.**

$$e_i=100\frac{\alpha_{code,i}-\alpha_{measured,i}}{\alpha_{measured,i}},\qquad APE_i=|e_i|.$$

dB/km 감쇠계수 자체의 상대오차이다. 전송 후 광전력 오차와 구별한다. 분산 기준 2.1/3.2/3.7은 논문의 FEM 값이며, 손실 0.128/0.091은 cutback 측정값이다.


In [ ]:
@dataclass(frozen=True)
class DNANFGeometry:
    core_radius_um: float = 14.75
    membrane_thickness_um: float = 0.50
    outer_tube_diameter_um: float = 31.05
    middle_tube_diameter_um: float = 23.75
    inner_tube_diameter_um: float = 7.70
    tube_count: int = 5
    nesting_order: int = 2

    @property
    def core_radius_m(self):
        return self.core_radius_um*1e-6

    @property
    def membrane_thickness_m(self):
        return self.membrane_thickness_um*1e-6

    @property
    def perimeter_gap_m(self):
        theta = np.pi/self.tube_count
        return (
            2.0*self.core_radius_m*np.sin(theta)
            - self.outer_tube_diameter_um*1e-6*(1.0-np.sin(theta))
        )

def hasan_radius_coefficients(geometry):
    radius_to_gap = geometry.core_radius_m/geometry.perimeter_gap_m
    n = geometry.tube_count
    nesting = geometry.nesting_order
    a0, a1 = 0.097041, 1.095
    b0, b1, b2, b3 = 0.76246, 0.007584, 0.002, 0.012
    f1 = a1*np.exp(a0/radius_to_gap)
    f2 = (
        b1*n*np.exp(b0/radius_to_gap)-b2*n+b3
        +0.0045*np.exp(-4.1589/(nesting*radius_to_gap))
    )
    return float(f1), float(f2)

def effective_radius_m(wavelength_m, geometry, f1, f2):
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    return f1*geometry.core_radius_m*(
        1.0-f2*wavelength_m**2/
        (geometry.core_radius_m*geometry.membrane_thickness_m)
    )

def effective_index(wavelength_m, geometry, f1, f2):
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    reff = effective_radius_m(wavelength_m, geometry, f1, f2)
    return 1.0-0.125*(
        U01*wavelength_m/(np.pi*reff)
    )**2

def chromatic_dispersion_ps_nm_km(
    wavelength_m, geometry, derivative_step_nm=0.20
):
    f1, f2 = hasan_radius_coefficients(geometry)
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    h = derivative_step_nm*1e-9
    def n_eff(offset):
        return effective_index(
            wavelength_m+offset, geometry, f1, f2
        )
    d2n = (
        -n_eff(2*h)+16*n_eff(h)-30*n_eff(0.0)
        +16*n_eff(-h)-n_eff(-2*h)
    )/(12*h**2)
    return -(wavelength_m/C0)*d2n/1e-6

def silica_index_sellmeier(wavelength_m):
    wavelength_um = np.asarray(wavelength_m, dtype=float)*1e6
    wavelength_sq = wavelength_um**2
    b = np.array([0.6961663, 0.4079426, 0.8974794])
    c_um = np.array([0.0684043, 0.1162414, 9.896161])
    n_sq = np.ones_like(wavelength_sq)
    for bi, ci in zip(b, c_um):
        n_sq += bi*wavelength_sq/(wavelength_sq-ci**2)
    return np.sqrt(n_sq)

def capillary_bouncing_ray_loss_db_km(wavelength_m, geometry):
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    k0 = 2*np.pi/wavelength_m
    radius = geometry.core_radius_m
    wall = geometry.membrane_thickness_m
    kappa = U01/radius
    silica_n = silica_index_sellmeier(wavelength_m)
    sigma = k0*np.sqrt(silica_n**2-1.0)
    phase = sigma*wall
    te_den = (
        4*np.cos(phase)**2
        +(kappa/sigma+sigma/kappa)**2*np.sin(phase)**2
    )
    tm_den = (
        4*np.cos(phase)**2
        +(silica_n**2*kappa/sigma
          +sigma/(silica_n**2*kappa))**2*np.sin(phase)**2
    )
    alpha_te = 2*U01/(radius**2*k0*te_den)
    alpha_tm = 2*U01/(radius**2*k0*tm_den)
    return 0.5*(alpha_te+alpha_tm)*DB_PER_NEPER*1000.0


In [ ]:
geometry = DNANFGeometry()
wl_d = np.array([1310.,1550.,1700.])
raw_d = chromatic_dispersion_ps_nm_km(wl_d*1e-9,geometry)
wl_loss = np.array([1310.,1550.])
raw_loss = capillary_bouncing_ray_loss_db_km(wl_loss*1e-9,geometry)
# Compare only after raw predictions exist; references never enter model functions.
ref_d = np.array([2.1,3.2,3.7])
ref_loss = np.array([.128,.091])
loss_df = pd.DataFrame({'wavelength_nm':wl_loss,'measured_total_db_km':ref_loss,
    'legacy_proxy_db_km':raw_loss,'signed_error_percent':signed_error_percent(raw_loss,ref_loss),
    'absolute_error_percent':np.abs(signed_error_percent(raw_loss,ref_loss))})
d_df = pd.DataFrame({'wavelength_nm':wl_d,'paper_FEM_ps_nm_km':ref_d,
    'raw_geometry_ps_nm_km':raw_d,'absolute_error_percent':np.abs(signed_error_percent(raw_d,ref_d))})
table('01_loss_errors',loss_df)
table('02_dispersion_errors',d_df)
print('Loss metrics:',error_metrics(raw_loss,ref_loss))
print('Dispersion metrics:',error_metrics(raw_d,ref_d))
fig,axes=plt.subplots(1,3,figsize=(14,4.2),layout='constrained')
x=np.arange(2)
axes[0].bar(x-.18,ref_loss,.36,label='Measured total loss',color='#167b83')
axes[0].bar(x+.18,raw_loss,.36,label='Legacy capillary proxy',color='#d85b43')
axes[0].set(yscale='log',xticks=x,xticklabels=['1310','1550'],xlabel='Wavelength (nm)',
            ylabel='Attenuation (dB/km)',title='Loss applicability audit')
axes[0].legend(fontsize=8)
axes[1].bar(['1310','1550'],np.abs(signed_error_percent(raw_loss,ref_loss)),color='#d85b43')
axes[1].set(ylabel='Absolute relative error (%)',xlabel='Wavelength (nm)',title='Loss error: order 10^8 %')
axes[1].ticklabel_format(axis='y',style='sci',scilimits=(0,0))
axes[2].plot(wl_d,ref_d,'o-',label='Paper FEM',color='#167b83')
axes[2].plot(wl_d,raw_d,'s--',label='Raw geometry',color='#d85b43')
axes[2].set(xlabel='Wavelength (nm)',ylabel='D (ps/nm/km)',title='Dispersion audit')
axes[2].legend()
figure('01_hcf_geometry_validation',fig)


**손실 모델 판단:** 원 코드의 절대손실을 GN·거리 계산에 넘기면 의미 없는 결과가 된다. 실측값으로 임의 정규화하면 보정점 재현일 뿐 독립 검증이 아니다. 정밀 손실 모델은 실제 중첩구조의 복소 $n_{eff}$로 leakage를 계산하고 surface scattering·microbend·gas 손실을 더해야 한다. 계수는 별도 섬유 데이터에서 결정하고, HCF2는 최종 검증에만 써야 한다.

아래 시스템에서 감쇠 0.091 dB/km를 사용하는 것은 **측정값을 입력한 시스템 추정**이다. geometry로 손실까지 독립 예측했다는 주장은 하지 않는다.


## 2. 기존 GN 검산 — 요청 논문의 전송 시뮬레이션 비교와 구별

Poggiolini 2012 Table I, 1550 nm, 32 GBd, Nyquist 32 GHz, 157 channels, 100 km/span. 기존 무한 유효길이 근사와 Eq.13을 비교한다.

유한 span 보정은 $\eta_{finite}=\eta_{legacy}(1-e^{-aL})^2$, $a=\alpha_{dB/km}\ln(10)/10$이다. 이 항으로 Eq.13과 일치하는 것은 **수식 일관성 검산**이다. 독립 SSFM 또는 실험 검증 오차가 0이 되었다는 뜻이 아니다.


In [ ]:
@dataclass(frozen=True)
class GNFiber:
    name: str
    attenuation_db_km: float
    dispersion_ps_nm_km: float
    gamma_per_w_km: float

def beta2_s2_per_km(dispersion_ps_nm_km, wavelength_nm=1550.0):
    wavelength_m = wavelength_nm*1e-9
    dispersion_si = np.asarray(dispersion_ps_nm_km, dtype=float)*1e-6
    return np.abs(
        -(wavelength_m**2/(2.0*np.pi*C0))*dispersion_si*1000.0
    )

def repository_closed_form_eta(
    fiber, symbol_rate_hz, spacing_hz, n_channels, n_spans=1
):
    beta2 = beta2_s2_per_km(fiber.dispersion_ps_nm_km)
    alpha_field = fiber.attenuation_db_km/8.685889638
    argument = (
        np.pi**2*beta2*symbol_rate_hz**2/(4.0*alpha_field)
        * n_channels**(2.0*symbol_rate_hz/spacing_hz)
    )
    return (
        n_spans*4.0*fiber.gamma_per_w_km**2
        /(27.0*np.pi*beta2*alpha_field*symbol_rate_hz**2)
        * np.arcsinh(argument)
    )

def poggiolini_eq13_eta(
    fiber, span_length_km, symbol_rate_hz, n_channels
):
    beta2 = beta2_s2_per_km(fiber.dispersion_ps_nm_km)
    alpha_field = fiber.attenuation_db_km/8.685889638
    leff = (
        1.0-np.exp(-2.0*alpha_field*span_length_km)
    )/(2.0*alpha_field)
    leff_asymptotic = 1.0/(2.0*alpha_field)
    bandwidth_hz = n_channels*symbol_rate_hz
    gnli_over_gwdm_cubed = (
        8.0/27.0*fiber.gamma_per_w_km**2*leff**2
        * np.arcsinh(
            0.5*np.pi**2*beta2*leff_asymptotic*bandwidth_hz**2
        )
        /(np.pi*beta2*leff_asymptotic)
    )
    return gnli_over_gwdm_cubed/symbol_rate_hz**2


In [ ]:
GN_FIBERS = [GNFiber('LPSCF',.165,20.4,.8),GNFiber('SMF',.2,16.5,1.3),GNFiber('NZDSF',.2,3.9,1.6)]
rows=[]
for f in GN_FIBERS:
    old_eta=repository_closed_form_eta(f,32e9,32e9,157)
    ref_eta=poggiolini_eq13_eta(f,100,32e9,157)
    # Same rounded field-attenuation constant as the original, for exact implementation audit.
    corrected=old_eta*(-np.expm1(-2*f.attenuation_db_km/8.685889638*100))**2
    rows.append({'fiber':f.name,'reference_Eq13_W_inv2':ref_eta,'legacy_W_inv2':old_eta,
        'finite_span_W_inv2':corrected,'legacy_APE_percent':abs(signed_error_percent(old_eta,ref_eta)),
        'finite_APE_percent':abs(signed_error_percent(corrected,ref_eta))})
gn_df=pd.DataFrame(rows)
table('03_gn_equation_consistency',gn_df)
check('Finite-length GN equals independently written Eq13 at Nyquist',np.max(gn_df.finite_APE_percent)<1e-8)
fig,axes=plt.subplots(1,2,figsize=(10.5,4),layout='constrained')
x=np.arange(3)
for offset,col,label,color in [(-.24,'reference_Eq13_W_inv2','Eq.13','#167b83'),
                               (0,'legacy_W_inv2','Legacy','#d85b43'),(.24,'finite_span_W_inv2','Finite span','#b48b20')]:
    axes[0].bar(x+offset,gn_df[col],.24,label=label,color=color)
axes[0].set(xticks=x,xticklabels=gn_df.fiber,ylabel='eta (1/W²)',title='GN equation comparison')
axes[0].legend(fontsize=8)
axes[1].bar(gn_df.fiber,gn_df.legacy_APE_percent,color='#d85b43')
axes[1].set(ylabel='Absolute relative error (%)',title='Legacy eta error')
figure('02_gn_equation_validation',fig)


In [ ]:
@dataclass(frozen=True)
class LaunchCase:
    name: str
    n_channels: int
    spacing_ghz: float
    span_km: float
    paper_popt_dbm: float

def repository_optimum_launch_dbm(case, noise_figure_db=6.0):
    fiber = next(f for f in GN_FIBERS if f.name == "SMF")
    eta = repository_closed_form_eta(
        fiber, 32e9, case.spacing_ghz*1e9, case.n_channels, 1
    )
    frequency_hz = C0/1550e-9
    noise_factor = 10.0**(noise_figure_db/10.0)
    gain = 10.0**(
        fiber.attenuation_db_km*case.span_km/10.0
    )
    ase_w = (
        H_PLANCK*frequency_hz*noise_factor*32e9*(gain-1.0)
    )
    popt_w = (ase_w/(2.0*eta))**(1.0/3.0)
    return 10.0*np.log10(popt_w)+30.0, popt_w, eta, ase_w


In [ ]:
PAPER_LAUNCH_CASES=[LaunchCase('RS-SMF 100 km',101,50.,100.,-.4),
                   LaunchCase('NY-SMF 100 km',157,32.,100.,-1.),
                   LaunchCase('NY-SMF 75 km',157,32.,75.,-2.6)]
rows=[]
for case in PAPER_LAUNCH_CASES:
    pred_dbm,pred_w,_,_=repository_optimum_launch_dbm(case)
    ref_w=dbm2w(case.paper_popt_dbm)
    rows.append({'case':case.name,'paper_dBm':case.paper_popt_dbm,'legacy_dBm':pred_dbm,
                 'delta_dB':pred_dbm-case.paper_popt_dbm,
                 'linear_power_APE_percent':abs(signed_error_percent(pred_w,ref_w))})
popt_df=pd.DataFrame(rows)
table('04_gn_optimum_power_examples',popt_df)
print('dBm의 백분율은 계산하지 않습니다. 선형 W의 APE와 dB 차이를 보고합니다.')


## 3. 첨부 그림: 출처 확인 상태와 디지타이즈

지정한 제목 *Analytical Modeling of Nonlinear Propagation in Uncompensated Optical Transmission Links*는 [Poggiolini et al., PTL 23, 742–744 (2011), DOI 10.1109/LPT.2011.2131125](https://iris.polito.it/handle/11583/2460606)이다. 첨부 그림의 네 변조방식·3종 섬유·32 GBd·채널 간격 범위는 [Carena et al., JLT 30, 1524–1539 (2012), DOI 10.1109/JLT.2012.2189198](https://opg.optica.org/abstract.cfm?uri=jlt-30-10-1524)의 공개 초록과 부합하지만 **실제 그림 캡션을 확인하지 못했으므로 출처와 패널별 조건을 확정하지 않는다.**

따라서 임의로 (a)=PSCF, (b)=SMF, (c)=NZDSF라고 입력하거나, NF·span·수신 필터·BER·FEC를 추측해서 오차를 계산하지 않는다. 현재 요청 2의 상태는 **INPUTS_UNVERIFIED**다.

아래는 첨부 이미지에 보이는 각 변조방식의 첫 번째 표시점(라벨 50) 12개를 수작업으로 읽은 데이터다. 전체 곡선이나 모든 표시점의 추출은 아니다. 좌표 오차를 ±3 px로 두며 약 ±7%의 거리 판독 오차가 생긴다. 이것은 모델 오차가 아니다. 기준 곡선/마커 중 무엇이 시뮬레이션인지는 캡션 확인이 필요하다.

원 논문의 PDF 또는 그림 캡션과 입력표가 확보되면 아래 `compare_paper_reference()`에 **출력에 피팅하지 않은 예측 CSV**를 넣어 한 그래프에 겹치고 MAPE/RMSE를 계산한다. span 수가 정수이면 그 양자화와 그림 판독 오차도 같이 해석해야 한다.


In [ ]:
# Image dimensions 356 x 820; plot axes read from user-provided image.
pixel_points=[('a','DP-BPSK',63,27),('a','DP-QPSK',115,58),('a','DP-8QAM',167,101),('a','DP-16QAM',220,139),
              ('b','DP-BPSK',63,339),('b','DP-QPSK',115,367),('b','DP-8QAM',167,414),('b','DP-16QAM',220,449),
              ('c','DP-BPSK',63,637),('c','DP-QPSK',115,665),('c','DP-8QAM',167,712),('c','DP-16QAM',220,747)]
axis_y={'a':(19.5,234.5),'b':(295.,509.5),'c':(570.5,786.)}
rows=[]
for panel,mod,x,y in pixel_points:
    top,bottom=axis_y[panel]
    def y_to_km(v): return 10**(np.log10(20000)-(v-top)/(bottom-top)*2)
    rows.append({'panel':panel,'modulation':mod,'marker_label':'50','x_px':x,'y_px':y,
                 'SE_bit_s_Hz':1+(x-63)/261*5,'reference_km':y_to_km(y),
                 'reference_low_km':y_to_km(y+3),'reference_high_km':y_to_km(y-3),
                 'reference_type':'unclassified_visible_marker','status':'INPUTS_UNVERIFIED'})
paper_ref=pd.DataFrame(rows)
table('05_attachment_partial_digitization',paper_ref)
fig,ax=plt.subplots(figsize=(8.4,4.5),layout='constrained')
for panel,grp in paper_ref.groupby('panel'):
    ax.errorbar(grp.SE_bit_s_Hz,grp.reference_km,
        yerr=[grp.reference_km-grp.reference_low_km,grp.reference_high_km-grp.reference_km],
        fmt='o--',capsize=3,label=f'Panel ({panel}), marker label 50 only')
ax.set(yscale='log',xlabel='Spectral efficiency (bit/s/Hz)',ylabel='Visible marker reach (km)',
       title='Attachment reading: selected reference points only')
ax.legend(fontsize=9)
figure('03_attachment_reference_only',fig)

PAPER_STATUS={'status':'INPUTS_UNVERIFIED','independent_error_percent':None,
    'required':['exact_paper_and_figure','panel_fibers_alpha_D_gamma','wavelength_nm','span_km',
                'NF_dB','Nch_or_bandwidth','Rs_and_spacing','Tx_Rx_filters_and_B2B_penalty',
                'BER_threshold_and_FEC_SE_definition','amplifier_loss_model','simulation_marker_identity'],
    'reason':'Attached plot caption and complete same-input conditions unavailable.'}
(RESULTS/'paper_validation_status.json').write_text(json.dumps(PAPER_STATUS,ensure_ascii=False,indent=2))

def compare_paper_reference(reference, predictions, verified_inputs):
    missing=[k for k in PAPER_STATUS['required'] if k not in verified_inputs or verified_inputs[k] is None]
    if missing: raise ValueError('동일 입력조건을 먼저 확인하십시오: '+', '.join(missing))
    if not verified_inputs.get('reference_outputs_excluded_from_fitting',False):
        raise ValueError('참조 출력이 피팅에서 제외되었는지 확인해야 합니다.')
    keys=['panel','modulation','marker_label']
    merged=reference.merge(predictions,on=keys,validate='one_to_one',how='left')
    if merged['code_km'].isna().any(): raise ValueError('일부 참조점의 예측 code_km가 없습니다.')
    if not np.all(np.isfinite(merged.code_km)&(merged.code_km>0)):
        raise ValueError('code_km는 양의 유한한 거리여야 합니다.')
    merged['signed_error_percent']=signed_error_percent(merged.code_km,merged.reference_km)
    merged['absolute_error_percent']=np.abs(merged.signed_error_percent)
    stats=[]
    fig,axes=plt.subplots(1,3,figsize=(14,4.4),layout='constrained')
    for ax,(panel,group) in zip(axes,merged.groupby('panel')):
        for mod,g in group.groupby('modulation'):
            line=ax.plot(g.SE_bit_s_Hz,g.code_km,'x--',label=mod+' code')[0]
            ax.errorbar(g.SE_bit_s_Hz,g.reference_km,
                        yerr=[g.reference_km-g.reference_low_km,g.reference_high_km-g.reference_km],
                        fmt='o',color=line.get_color(),capsize=3,label=mod+' reference')
        ax.set(yscale='log',xlabel='SE (bit/s/Hz)',ylabel='Reach (km)',title='Panel '+panel)
        ax.legend(fontsize=7)
        stats.append({'panel':panel,**error_metrics(group.code_km,group.reference_km)})
    table('paper_same_input_errors',merged)
    table('paper_same_input_metrics',pd.DataFrame(stats))
    figure('paper_same_input_overlay',fig)
    return merged

# After source verification, supply independently predicted distances keyed by the same points:
# predictions=pd.read_csv('paper_predictions.csv')  # panel,modulation,marker_label,code_km
# compare_paper_reference(paper_ref,predictions,verified_inputs)
print('Requested paper simulation error: NOT COMPUTED. It is not 0%.')


## 4. 시스템 입력과 가정

SCUBA110 웹 사양은 G.654.B/D 대응이며 **G.654.E 지상용 옵션도 제공**한다. 여기서는 공개된 SCUBA110 광학 사양으로 비교한다. G.654.E 등급 전체의 대표값 또는 납품 측정값으로 간주하지 않는다. 감쇠 0.151, 분산 22는 공개 상한을 대입한 시나리오이며 enhanced 감쇠 0.147도 별도 계산한다. $A_{eff}=110$ μm², silica $n_2=2.2\times10^{-20}$ m²/W(가정)로 $\gamma=2\pi n_2/(\lambda A_{eff})$를 구한다.

HCF: 측정 감쇠 0.091 dB/km, 원 geometry 모델의 1550 nm 분산을 입력한다. **HCF2의 정량적 $\gamma$가 확보되지 않았으므로 0.001 W⁻¹km⁻¹를 가정**하고 0/0.0001/0.001/0.01을 스캔한다. 이는 실측 범위 또는 신뢰구간이 아니다. 모든 채널에 1550 nm 파라미터를 동일 적용하는 C-band 근사다.

Hedeboe 코드의 4.3 THz, 100 GBd, 112.5 GHz, NF 4.5 dB, 12 fiber pairs, 회로손실 10%, 18 dBm 총 WDM 광출력 상한, 2 dB 비선형 최적전력 backoff를 유지한다. **효율표가 없어서 재구성 곡면 대신 고정 2.3%**를 쓴다. 채널 수는 물리적으로 정수인 floor(4.3 THz/112.5 GHz)=38로 바꿨다.

추가 설계 가정: 전기전력 50 W/리피터, 100개 유효 증폭구간, TRX SNR 20 dB, span 추가 삽입손실 0 dB, BER 기준 $10^{-3}$, 설계여유 1 dB, code rate $1/1.2$. 특정 FEC가 이 BER에서 동작한다는 주장은 하지 않는다. 실제 FEC·DSP·PMD·모드결합·필터·splice·EDFA gain 범위·ISRS·신호-ASE 비선형 상호작용은 미검증이다.


In [ ]:
@dataclass(frozen=True)
class Fiber:
    name: str
    attenuation_db_km: float
    dispersion_ps_nm_km: float
    gamma_per_w_km: float
    extra_span_loss_db: float = 0.

@dataclass(frozen=True)
class System:
    wavelength_nm: float = 1550.
    Rs_hz: float = 100e9
    spacing_hz: float = 112.5e9
    bandwidth_hz: float = 4.3e12
    n_channels: int = 38
    nf_db: float = 4.5
    spatial_pairs: int = 12
    circuit_fraction: float = .10
    eta_eo: float = .023
    total_wdm_hard_max_dbm: float = 18.
    nonlinear_backoff_db: float = 2.
    trx_snr_db: float = 20.
    repeater_power_w: float = 50.
    n_spans: int = 100
    route_km: float = 8000.
    pfe_voltage_v: float = 18000.
    cable_ohm_km: float = 1.
    target_ber: float = 1e-3
    code_rate: float = 1/1.2
    design_margin_db: float = 1.

S=System()
gamma_scuba=2*np.pi*2.2e-20/(S.wavelength_nm*1e-9*110e-12)*1000
SCUBA=Fiber('SCUBA110',.151,22.,gamma_scuba)
HCF=Fiber('HCF DNANF',.091,float(raw_d[1]),.001)
FIBERS=[SCUBA,HCF]
COLORS={'SCUBA110':'#d85b43','HCF DNANF':'#167b83'}
table('06_fiber_inputs',pd.DataFrame([asdict(f) for f in FIBERS]))
table('07_system_inputs',pd.DataFrame([{'parameter':k,'value':v} for k,v in asdict(S).items()]))
provenance=pd.DataFrame([
 ['SCUBA attenuation','.151 dB/km','Lightera standard maximum; .147 enhanced sensitivity'],
 ['SCUBA D','22 ps/nm/km','Lightera maximum, scenario input'],
 ['SCUBA Aeff','110 um2','Lightera specification'],
 ['SCUBA n2','2.2e-20 m2/W','assumption; gamma derived'],
 ['HCF attenuation','.091 dB/km','Petrovich HCF2 measured at 1550 nm'],
 ['HCF D',str(HCF.dispersion_ps_nm_km),'legacy geometry prediction, not fitted'],
 ['HCF gamma','.001 1/W/km','assumption, not measured HCF2'],
 ['EDFA efficiency','.023','constant scenario from original power notebook; no efficiency table'],
 ['BER / FEC','1e-3 / rate 1/1.2','design assumptions, no FEC decoder simulation'],
 ['Nrep convention','Nrep=Nsp=100','preserved effective amplifier-section convention; not submerged BOM'],
],columns=['input','value','provenance'])
table('08_provenance',provenance)
(RESULTS/'configuration.json').write_text(json.dumps({'system':asdict(S),'fibers':[asdict(f) for f in FIBERS],
    'paper_status':PAPER_STATUS,'model':'finite-span GN + generalized droop; AWGN demapper'},ensure_ascii=False,indent=2))


## 5. 유한 span GN + Generalized Droop + 전력 제약

span당 잡음 $A=h\nu F R_s(G-1)$, $P_{NLI}=\eta P_{ch}^3$. 여기서 $P_{ch}$는 **한 WDM 채널의 두 편광 합산 전력**이고 noise bandwidth는 $R_s$다. 0.1 nm OSNR과 혼동하지 않는다.

$$GSNR_{GN}^{-1}=N\left(\frac{A}{P_{ch}}+\eta P_{ch}^2\right)+SNR_{TRX}^{-1},$$
$$GSNR_{GD}^{-1}=\left[(1+A/P_{ch})(1+\eta P_{ch}^2)\right]^N-1+SNR_{TRX}^{-1}.$$

같은 잡음항을 GN과 GD로 **두 번 더하지 않는다**. 최종 시스템 계산은 원 Hedeboe 흐름에 맞춰 GD를 쓰고 GN은 비교 곡선으로만 쓴다. 기본 GN은 변조별 NLI 차이를 모델링하지 않는다. BPSK/QPSK 차이는 여기서 AWGN 판정과 정보율에서 나온다. 낮은 HCF 분산에 대한 GN 정확도는 별도 SSFM/EGN 검증이 필요하다.

$$P_{WDM,max}=\min\left(\frac{(1-\epsilon)\eta_{eo}P_{rep}}{2M_{sp}},10^{(18-30)/10}\right),\quad P_{ch,max}=P_{WDM,max}/N_{ch}.$$

**전력이 같으면 두 섬유의 광출력 상한도 같다.** HCF의 낮은 $\gamma$는 비선형 최적전력의 상한을 완화한다. 총 WDM 18 dBm를 채널당 18 dBm로 입력하면 안 된다.

PFE 비교는 별도다:
$$P_{rep}=\frac{V^2}{4R_0L_{tot}N},\qquad V_{required}=2\sqrt{R_0L_{tot}NP_{rep}}.$$
거리 변화 시 $V$ 고정과 $P_{rep}$ 고정은 동시에 유지되지 않는다. MPT에서 부하와 도체가 같은 전력을 소모하므로 부하 5 kW는 공급원 출력 10 kW에 해당한다.


In [ ]:
def span_noise(fiber,span_km,system=S):
    if span_km<=0 or fiber.attenuation_db_km<=0 or fiber.dispersion_ps_nm_km==0:
        raise ValueError('Positive span/attenuation and nonzero D required by this GN approximation.')
    a=fiber.attenuation_db_km*np.log(10)/10  # POWER attenuation, 1/km
    b2=float(beta2_s2_per_km(fiber.dispersion_ps_nm_km,system.wavelength_nm))
    leff=-np.expm1(-a*span_km)/a
    linf=1/a
    arg=.5*np.pi**2*b2*linf*system.Rs_hz**2*system.n_channels**(2*system.Rs_hz/system.spacing_hz)
    eta=8/27*fiber.gamma_per_w_km**2*leff**2*np.arcsinh(arg)/(np.pi*b2*linf*system.Rs_hz**2)
    gain_db=fiber.attenuation_db_km*span_km+fiber.extra_span_loss_db
    ase=H_PLANCK*C0/(system.wavelength_nm*1e-9)*db2lin(system.nf_db)*system.Rs_hz*np.expm1(gain_db*np.log(10)/10)
    return float(ase),float(eta),gain_db

def gsnr(fiber,launch_w,span_km,n_spans,system=S,model='GD'):
    p=np.asarray(launch_w,dtype=float)
    if np.any(p<=0) or n_spans<1: raise ValueError('Positive power and at least one span required.')
    ase,eta,_=span_noise(fiber,span_km,system)
    trx_inverse=0. if np.isinf(system.trx_snr_db) else float(db2lin(-system.trx_snr_db))
    if model=='GD':
        exponent=n_spans*(np.log1p(ase/p)+np.log1p(eta*p*p))
        inv=np.expm1(np.minimum(exponent,700.))
    elif model=='GN': inv=n_spans*(ase/p+eta*p*p)
    else: raise ValueError("model must be 'GD' or 'GN'")
    return 1/(inv+trx_inverse)

def power_ceiling(repeater_w,system=S):
    optical=(1-system.circuit_fraction)*system.eta_eo*repeater_w/(2*system.spatial_pairs)
    return min(optical,float(dbm2w(system.total_wdm_hard_max_dbm)))/system.n_channels

def pfe_repeater_power(route_km,n_spans,system=S):
    return system.pfe_voltage_v**2/(4*system.cable_ohm_km*route_km*n_spans)

def required_mpt_voltage(route_km,n_spans,repeater_w,system=S):
    return 2*np.sqrt(system.cable_ohm_km*route_km*n_spans*repeater_w)

def optimum_operating_point(fiber,span_km,n_spans,repeater_w,system=S):
    ase,eta,gain_db=span_noise(fiber,span_km,system)
    ceiling=power_ceiling(repeater_w,system)
    if eta==0:
        p_nl=np.inf
    else:
        # Derivative of (1+A/P)(1+eta P²): 2 eta P³ + A eta P² - A = 0.
        # Additive GN optimum is an analytic upper bracket for the GD optimum.
        p_gn=(ase/(2*eta))**(1/3)
        p_nl=brentq(lambda q:2*eta*q**3+ase*eta*q*q-ase,p_gn*1e-9,p_gn,xtol=1e-16)
    backed=p_nl/float(db2lin(system.nonlinear_backoff_db))
    p_use=min(ceiling,backed)
    snr=float(gsnr(fiber,p_use,span_km,n_spans,system))
    constraint='electrical/hardware' if ceiling<=backed else 'nonlinear optimum + backoff'
    return {'launch_dBm':float(w2dbm(p_use)),'launch_W':p_use,
            'total_WDM_dBm':float(w2dbm(p_use*system.n_channels)),
            'ceiling_ch_dBm':float(w2dbm(ceiling)),'NL_optimum_dBm':float(w2dbm(p_nl)),
            'GSNR_dB':float(lin2db(snr)),'GSNR_linear':snr,'gain_dB':gain_db,
            'limiting_constraint':constraint}

ceiling=power_ceiling(S.repeater_power_w,S)
print(f'Common optical ceiling: total {w2dbm(ceiling*S.n_channels):.3f} dBm, per channel {w2dbm(ceiling):.3f} dBm')
print(f'Total delivered repeater electrical power: {S.n_spans*S.repeater_power_w:.0f} W')
check('Total WDM power respects hardware ceiling',ceiling*S.n_channels<=dbm2w(S.total_wdm_hard_max_dbm))
check('38 integer channels fit within available slots',S.n_channels*S.spacing_hz<=S.bandwidth_hz)


## 6. DP-BPSK·DP-QPSK: BER, AIR, payload 정의

$s=GSNR$는 편광당 복소 심벌 $E_s/N_0$와 같다(신호와 잡음을 모두 두 편광으로 합산해도 비율 동일).
$$BER_{BPSK}=\tfrac12\operatorname{erfc}(\sqrt{s}),\qquad BER_{QPSK}=\tfrac12\operatorname{erfc}(\sqrt{s/2}).$$

AWGN의 유한 constellation 정보율을 Gauss–Hermite 적분으로 계산한다:
$$I_B(s)=1-\mathbb E_{Z\sim\mathcal N(0,1)}\log_2[1+e^{-4s-\sqrt{8s}Z}],\quad I_Q(s)=2I_B(s/2).$$
$$T_{AIR}=2N_{ch}R_s I(s),\qquad T_{Shannon}=2N_{ch}R_s\log_2(1+s).$$
균등 Gray BPSK/QPSK의 이상적 AWGN demapper에서는 이 MI가 bitwise GMI와 일치한다. 광전송 신호에서 직접 측정한 GMI는 아니다. Shannon을 변조 상한으로 자르는 근사 대신 실제 유한 constellation 적분을 사용한다.

별도로 설계 payload $T_{payload}=2N_{ch}R_s\log_2(M)R_c$는 `GSNR ≥ BER 목표의 이론 요구 GSNR + 여유`일 때만 인정한다. 실제 FEC 디코딩 성공률이 아니라 설계 판정이다. 같은 baud·채널·code rate에서 QPSK payload는 BPSK의 2배이며 요구 GSNR은 3.01 dB 높다.

원 Hedeboe `capacity_bps()`는 $N_{ch}R_s\log_2(1+s)$ 형태였다. 본 노트북은 **DP 기준을 명시해 편광 2배를 적용**한다. 모든 그래프 throughput은 fiber당·단방향이며 전체 12-pair 케이블 단방향 값은 12배다. 양방향 합계가 필요하면 그 값의 2배다.


In [ ]:
MODULATIONS={'DP-BPSK':2,'DP-QPSK':4}
GH_X,GH_W=hermgauss(80)
def bpsk_mi(snr):
    s=np.asarray(snr,dtype=float)[...,None]
    z=np.sqrt(2)*GH_X
    softplus=np.logaddexp(0.,-4*s-np.sqrt(8*s)*z)/np.log(2)
    return np.clip(1-np.sum(softplus*GH_W/np.sqrt(np.pi),axis=-1),0.,1.)
def constellation_air(snr,modulation,system=S):
    if modulation=='DP-BPSK': perpol=bpsk_mi(snr)
    elif modulation=='DP-QPSK': perpol=2*bpsk_mi(np.asarray(snr)/2)
    else: raise ValueError('Only BPSK/QPSK AWGN AIR implemented.')
    return 2*system.n_channels*system.Rs_hz*perpol
def ber_awgn(snr,modulation):
    return .5*erfc(np.sqrt(np.asarray(snr)/(1 if modulation=='DP-BPSK' else 2)))
def required_snr(modulation,system=S):
    base=erfcinv(2*system.target_ber)**2*(1 if modulation=='DP-BPSK' else 2)
    return base*db2lin(system.design_margin_db)
def payload_ceiling(modulation,system=S):
    return 2*system.n_channels*system.Rs_hz*np.log2(MODULATIONS[modulation])*system.code_rate
def payload_at_snr(snr,modulation,system=S):
    return np.where(np.asarray(snr)>=required_snr(modulation,system),payload_ceiling(modulation,system),0.)
mod_df=pd.DataFrame([{'modulation':m,'required_GSNR_with_margin_dB':float(lin2db(required_snr(m))),
    'raw_gross_Tbps_per_fiber':2*S.n_channels*S.Rs_hz*np.log2(M)/1e12,
    'design_payload_Tbps_per_fiber':payload_ceiling(m)/1e12} for m,M in MODULATIONS.items()])
table('09_modulation_thresholds',mod_df)


## 7. launch power → GSNR·throughput

8,000 km / 100 spans = 80 km/span. 연속 곡선은 모형의 전력 응답이며, 회색 영역은 동일 50 W 리피터 조건에서 공급할 수 없는 광출력이다. 동그라미는 2 dB backoff까지 적용한 실제 선택점이다. 0 dB backoff 결과는 뒤의 민감도 표에서 비교한다.


In [ ]:
p_dbm=np.linspace(-18,12,401)
p_w=dbm2w(p_dbm)
span=S.route_km/S.n_spans
launch_rows=[]
operating=[]
fig,axes=plt.subplots(2,2,figsize=(12.5,8.1),layout='constrained')
for f in FIBERS:
    sn=gsnr(f,p_w,span,S.n_spans,S,'GD')
    sn_gn=gsnr(f,p_w,span,S.n_spans,S,'GN')
    op=optimum_operating_point(f,span,S.n_spans,S.repeater_power_w)
    for m in MODULATIONS:
        operating.append({'fiber':f.name,'modulation':m,**op,
            'BER':float(ber_awgn(op['GSNR_linear'],m)),
            'AWGN_AIR_Tbps':float(constellation_air(op['GSNR_linear'],m)/1e12),
            'design_payload_Tbps':float(payload_at_snr(op['GSNR_linear'],m)/1e12)})
    co=COLORS[f.name]
    axes[0,0].plot(p_dbm,lin2db(sn),color=co,label=f.name+' GD')
    axes[0,0].plot(p_dbm,lin2db(sn_gn),':',color=co,label=f.name+' additive GN')
    axes[0,0].plot(op['launch_dBm'],op['GSNR_dB'],'o',color=co)
    sh=2*S.n_channels*S.Rs_hz*np.log2(1+sn)/1e12
    axes[0,1].plot(p_dbm,sh,':',color=co,label=f.name+' Shannon')
    for m,style in [('DP-BPSK','--'),('DP-QPSK','-')]:
        air=constellation_air(sn,m)/1e12
        payload=payload_at_snr(sn,m)/1e12
        axes[0,1].plot(p_dbm,air,style,color=co,label=f.name+' '+m)
        axes[1,0].plot(p_dbm,payload,style,color=co,label=f.name+' '+m)
        axes[1,1].semilogy(p_dbm,np.maximum(ber_awgn(sn,m),1e-12),style,color=co,label=f.name+' '+m)
        for i in range(len(p_dbm)):
            launch_rows.append({'fiber':f.name,'modulation':m,'launch_dBm':p_dbm[i],
                'GSNR_GD_dB':float(lin2db(sn[i])),'GSNR_GN_dB':float(lin2db(sn_gn[i])),
                'AIR_Tbps_per_fiber':air[i],'Shannon_Tbps_per_fiber':sh[i],
                'design_payload_Tbps':payload[i],'BER':float(ber_awgn(sn[i],m)),
                'electrical_hardware_feasible':p_w[i]<=ceiling,
                'backoff_policy_feasible':p_w[i]<=dbm2w(op['NL_optimum_dBm']-S.nonlinear_backoff_db)})
for ax in axes.flat:
    ax.axvspan(float(w2dbm(ceiling)),12,color='#64748b',alpha=.13)
    ax.axvline(float(w2dbm(ceiling)),color='#475569',lw=1)
    ax.set(xlabel='Launch power (dBm/channel, both polarizations)',xlim=(-18,12))
    ax.legend(fontsize=7.5,loc='best')
axes[0,0].set(ylabel='GSNR (dB)',title='8,000 km: GD and additive GN',ylim=(-10,22))
axes[0,1].set(ylabel='AWGN information rate (Tb/s/fiber)',title='Shannon and finite-constellation AIR')
axes[1,0].set(ylabel='Design payload (Tb/s/fiber)',title='Fixed code rate; BER + 1 dB margin gate')
axes[1,1].set(ylabel='Pre-FEC AWGN BER',title='Modulation performance',ylim=(1e-12,1))
axes[1,1].axhline(S.target_ber,color='gray',ls=':')
figure('04_launch_gsnr_throughput',fig)
table('10_launch_sweep',pd.DataFrame(launch_rows),False)
op_df=pd.DataFrame(operating)
table('11_operating_points_8000km',op_df)


## 8. 동일 리피터 전력의 최대 전송거리

100개 유효 증폭구간과 50 W/리피터를 고정하여 **동일한 부하 전력 5,000 W**로 비교한다. 동일 span에서의 비교가 아니라 총거리를 늘리면서 span을 늘려 BER+여유 조건의 경계를 찾는다. 채널 수·변조·code rate는 고정되어 비교 대상 payload가 같다.

거리 계산은 연속적인 span 길이의 모형 최적화다. 실제 바다 구간 배치·리피터 개수 산정에는 사용하지 않는다. 원 보고서와 같은 $N_{rep}=N_{sp}$ 관례이며 terminal까지 포함한 유효 증폭구간 수이므로 해저 리피터 BOM과 다를 수 있다.

`required_PFE_kV`는 이 부하 전력을 해당 거리에서 MPT로 공급하기 위한 전압이다. 18 kV를 초과하면 별도 18 kV 제한 실험으로 확인한다. 높은 gain의 NF·전기광 효율이 실제로 유지되는지는 효율표가 있어야 검증할 수 있다.


In [ ]:
def maximum_reach(fiber,modulation,system=S,power_mode='fixed_repeater',n_spans=None):
    n=system.n_spans if n_spans is None else int(n_spans)
    threshold=float(required_snr(modulation,system))
    def at_span(span):
        rep=(system.repeater_power_w if power_mode=='fixed_repeater'
             else pfe_repeater_power(span*n,n,system))
        return optimum_operating_point(fiber,span,n,rep,system),rep
    if power_mode not in ['fixed_repeater','fixed_pfe']: raise ValueError('Unknown power mode.')
    def residual(span): return at_span(span)[0]['GSNR_linear']-threshold
    low,high=1.,100.
    if residual(low)<0: raise ValueError('Threshold is infeasible even at 1 km spans.')
    while residual(high)>0 and high<1000: high*=1.5
    if residual(high)>0: raise ValueError('Reach exceeds model search bracket; do not report as finite optimum.')
    span=brentq(residual,low,high,xtol=1e-7)
    op,rep=at_span(span)
    km=span*n
    return {'fiber':fiber.name,'modulation':modulation,'power_mode':power_mode,
            'Nsp':n,'span_km':span,'reach_km':km,'repeater_W':rep,'total_delivered_W':rep*n,
            'required_PFE_kV':required_mpt_voltage(km,n,rep,system)/1000,
            'payload_Tbps_per_fiber':payload_ceiling(modulation,system)/1e12,**op}

reach_df=pd.DataFrame([maximum_reach(f,m,S,mode) for mode in ['fixed_repeater','fixed_pfe']
                     for f in FIBERS for m in MODULATIONS])
gain_rows=[]
for mode,g in reach_df.groupby('power_mode'):
    for m in MODULATIONS:
        sg=float(g[(g.fiber==SCUBA.name)&(g.modulation==m)].reach_km.iloc[0])
        hg=float(g[(g.fiber==HCF.name)&(g.modulation==m)].reach_km.iloc[0])
        gain_rows.append({'power_mode':mode,'modulation':m,'SCUBA_km':sg,'HCF_km':hg,
                          'HCF_distance_gain_percent':100*(hg/sg-1)})
gain_df=pd.DataFrame(gain_rows)
table('12_maximum_reach',reach_df)
table('13_distance_gain',gain_df)
fig,axes=plt.subplots(1,2,figsize=(11.5,4.6),layout='constrained')
for ax,mode,title in zip(axes,['fixed_repeater','fixed_pfe'],['50 W/repeater, 100 spans','18 kV PFE, 100 spans']):
    sub=reach_df[reach_df.power_mode==mode]
    x=np.arange(2)
    for delta,f in [(-.19,SCUBA),(.19,HCF)]:
        vals=[float(sub[(sub.fiber==f.name)&(sub.modulation==m)].reach_km.iloc[0]) for m in MODULATIONS]
        bars=ax.bar(x+delta,np.array(vals)/1000,.38,label=f.name,color=COLORS[f.name])
        ax.bar_label(bars,labels=[f'{v/1000:.2f}' for v in vals],padding=3,fontsize=9)
    ax.set(xticks=x,xticklabels=list(MODULATIONS),ylabel='Maximum modeled reach (1,000 km)',title=title)
    ax.set_ylim(0,float(reach_df.reach_km.max())/1000*1.17)
    ax.legend(fontsize=9)
figure('05_equal_power_reach',fig)
print('Distance gain percentages are fiber-to-fiber scenario gains, not validation errors.')


## 9. 전력·손실·비선형성 민감도와 같은 경로의 span 수

HCF 이점을 감쇠와 $\gamma$로 분리하기 위해 대조군을 계산한다. `alpha only`는 SCUBA의 D·γ를 유지하고 감쇠만 HCF 수준으로 바꾸며, `gamma only`는 SCUBA의 감쇠·D를 유지하고 γ만 HCF 가정값으로 바꾼다. 완전 HCF는 D도 달라지므로 각 효과의 합이 정확히 전체 이득이 되지는 않는다.

또한 실제 접속손실과 효율의 불확실성을 보기 위해 1/2 dB 추가 span 손실, 감쇠 0.12/0.15, γ 범위, 0 dB backoff를 계산한다. 이 표는 실측 신뢰구간이 아니다. 8,000 km에서 고정 18 kV로 가장 적은 유효 증폭구간 수도 찾는다.


In [ ]:
power_grid=np.linspace(10,120,35)
power_rows=[]
for f in FIBERS:
    for m in MODULATIONS:
        for rp in power_grid:
            row=maximum_reach(f,m,replace(S,repeater_power_w=float(rp)))
            power_rows.append(row)
power_df=pd.DataFrame(power_rows)
table('14_power_distance_sweep',power_df,False)
variants=[('SCUBA baseline',SCUBA,S),
          ('SCUBA enhanced alpha .147',replace(SCUBA,attenuation_db_km=.147),S),
          ('alpha only',replace(SCUBA,attenuation_db_km=HCF.attenuation_db_km),S),
          ('gamma only',replace(SCUBA,gamma_per_w_km=HCF.gamma_per_w_km),S),
          ('HCF baseline',HCF,S),
          ('HCF D = paper FEM 3.2',replace(HCF,dispersion_ps_nm_km=3.2),S),
          ('HCF extra loss 1 dB/span',replace(HCF,extra_span_loss_db=1.),S),
          ('HCF extra loss 2 dB/span',replace(HCF,extra_span_loss_db=2.),S),
          ('HCF alpha .12',replace(HCF,attenuation_db_km=.12),S),
          ('HCF alpha .15',replace(HCF,attenuation_db_km=.15),S),
          ('SCUBA backoff 0 dB',SCUBA,replace(S,nonlinear_backoff_db=0.)),
          ('HCF backoff 0 dB',HCF,replace(S,nonlinear_backoff_db=0.)),
          ('HCF efficiency 1.5%',HCF,replace(S,eta_eo=.015)),
          ('HCF efficiency 3%',HCF,replace(S,eta_eo=.03))]
variants += [(f'HCF gamma {g:g}',replace(HCF,gamma_per_w_km=g),S) for g in [0.,.0001,.001,.01]]
sens_rows=[]
for label,f,ss in variants:
    for m in MODULATIONS:
        sens_rows.append({'variant':label,**maximum_reach(f,m,ss)})
sens_df=pd.DataFrame(sens_rows)
table('15_sensitivity',sens_df)
fig,axes=plt.subplots(1,2,figsize=(12,4.9),layout='constrained')
for f in FIBERS:
    for m,style in [('DP-BPSK','--'),('DP-QPSK','-')]:
        g=power_df[(power_df.fiber==f.name)&(power_df.modulation==m)]
        axes[0].plot(g.repeater_W,g.reach_km/1000,style,color=COLORS[f.name],label=f.name+' '+m)
axes[0].set(xlabel='Electrical power per repeater (W)',ylabel='Reach (1,000 km)',title='100 spans; optical output ceiling enforced')
axes[0].legend(fontsize=8)
labels=['SCUBA baseline','alpha only','gamma only','HCF baseline','HCF extra loss 2 dB/span']
v=sens_df[(sens_df.modulation=='DP-QPSK')&sens_df.variant.isin(labels)].set_index('variant').loc[labels]
axes[1].barh(np.arange(len(v)),v.reach_km/1000,color=['#d85b43','#b48b20','#8584ad','#167b83','#64848b'])
axes[1].set(yticks=np.arange(len(v)),yticklabels=labels,xlabel='DP-QPSK reach (1,000 km)',title='Separate attenuation and gamma changes')
axes[1].invert_yaxis()
figure('06_power_sensitivity',fig)

count_rows=[]
for f in FIBERS:
    for m in MODULATIONS:
        for n in range(10,501):
            rep=pfe_repeater_power(S.route_km,n)
            op=optimum_operating_point(f,S.route_km/n,n,rep)
            if op['GSNR_linear']>=required_snr(m):
                count_rows.append({'fiber':f.name,'modulation':m,'route_km':S.route_km,'Nsp':n,
                    'span_km':S.route_km/n,'repeater_W':rep,'total_delivered_W':rep*n,**op})
                break
        else: raise RuntimeError('No feasible amplifier count in 10..500.')
table('16_minimum_sections_8000km_fixed18kV',pd.DataFrame(count_rows))


## 10. 수치 검산과 적용 범위

물리적으로 중요한 오류를 검사한다: 전력 단위/양편광, 유효길이, 같은 전력 상한, GD의 낮은 잡음 극한, 원 Hedeboe 구현과의 일치, AIR 상한, 거리 경계에서 BER 조건 만족. 원 코드의 감쇠 환산 4.343 반올림만 정확한 $10/\ln 10$으로 바뀌므로 소규모 차이가 예상된다.

**연구 활용:** 측정 α·검증된 D·γ·EDFA 효율을 교체해 시스템 설계 비교에 재사용할 수 있다. 그러나 현 geometry만으로 총손실·GSNR·capacity를 정밀 예측했다고 주장하면 안 된다. 낮은 D의 HCF, DP-BPSK에서 GN의 Gaussian 가정, 긴 span에서의 NF/gain/전기광 효율을 SSFM/EGN 및 독립 측정으로 확인해야 한다. 개선한 함수를 평가 데이터에 다시 맞춘 뒤 동일 데이터의 오차를 독립 검증이라고 보고하지 않는다.


In [ ]:
def original_snr_gd(Ps_w, Nrep, Ltot_km, fifo_db=0.0, gamma_xt_lin_km=0.0, p=None):
    Ps_w = np.asarray(Ps_w, dtype=float)
    h = 6.62607015e-34
    c_nm_s = 2.99792458e17
    f_hz = c_nm_s / p['lambda_nm']
    Nch = p['B_hz'] / p['df_hz']
    Pch = Ps_w / Nch
    alpha_lin = p['alpha_db_km'] / 4.343
    NF_lin = db2lin(p['NF_db'])
    beta2 = -p['D_s_km_nm'] * p['lambda_nm']**2 / (2*np.pi*c_nm_s)

    actual_span_km = Ltot_km / Nrep
    gain_db = p['alpha_db_km']*actual_span_km + 2*fifo_db
    G_lin = db2lin(gain_db)
    P_ASE = h*f_hz*NF_lin*p['Rs_hz']*(G_lin-1)
    chi_a = (1 + P_ASE/Pch)**-1

    L_eff = (1-np.exp(-alpha_lin*actual_span_km))/alpha_lin
    L_effa = 1/alpha_lin
    alpha_nli = (
        8/27 * p['gamma_w_km']**2 * L_eff**2
        * np.arcsinh(np.pi**2/2*abs(beta2)*L_effa*p['Rs_hz']**2
                     * Nch**(2*p['Rs_hz']/p['df_hz']))
        / (np.pi*abs(beta2)*L_effa*p['Rs_hz']**2)
    )
    snr_1r_nli = 1/(alpha_nli*Pch**2)
    if gamma_xt_lin_km == 0:
        snr_1r = snr_1r_nli
    else:
        snr_1r_xt = 1/(gamma_xt_lin_km*actual_span_km)
        snr_1r = 1/(1/snr_1r_nli + 1/snr_1r_xt)
    chi_r = (1 + 1/snr_1r)**-1
    return 1/((1/chi_a/chi_r)**Nrep - 1)


In [ ]:
p_legacy={'alpha_db_km':.15,'D_s_km_nm':21e-12,'gamma_w_km':.81,'NF_db':4.5,
          'B_hz':4.3e12,'df_hz':112.5e9,'Rs_hz':100e9,'lambda_nm':1550.}
ss_legacy=replace(S,n_channels=p_legacy['B_hz']/p_legacy['df_hz'],trx_snr_db=np.inf)
ff_legacy=Fiber('Regression fiber',.15,21.,.81)
pv=dbm2w(np.array([-7.,-3.,0.,3.]))
orig=original_snr_gd(pv*ss_legacy.n_channels,100,8000,p=p_legacy)
new=gsnr(ff_legacy,pv,80,100,ss_legacy)
gd_delta_db=np.max(np.abs(lin2db(new/orig)))
check('GD preserves original power-notebook result within 0.0001 dB',gd_delta_db<1e-4)
check('SCUBA gamma unit conversion near 0.811 /W/km',.80<SCUBA.gamma_per_w_km<.82)
sn_test=db2lin(np.linspace(-30,30,120))
for m,M in MODULATIONS.items():
    ai=constellation_air(sn_test,m)
    cap=2*S.n_channels*S.Rs_hz*np.log2(M)
    check(m+' finite constellation upper bound',np.max(ai)<=cap*(1+1e-12))
    check(m+' monotonic AWGN AIR',np.min(np.diff(ai))>=-cap*1e-9)
    check(m+' AWGN AIR high-SNR limit',ai[-1]/cap>.99999)
    base=required_snr(m,replace(S,design_margin_db=0.))
    check(m+' BER inversion',abs(float(ber_awgn(base,m))-S.target_ber)<1e-12)
check('QPSK requires twice BPSK Es/N0',abs(required_snr('DP-QPSK')/required_snr('DP-BPSK')-2)<1e-12)
f0=replace(HCF,gamma_per_w_km=0.)
ss0=replace(S,trx_snr_db=np.inf)
a0,_,_=span_noise(f0,80,ss0)
expected=1/np.expm1(100*np.log1p(a0/.001))
check('Zero gamma gives ASE-only droop',np.isclose(gsnr(f0,.001,80,100,ss0),expected,rtol=1e-12))
sn_gd=gsnr(f0,.001,1,1,ss0)
sn_gn=gsnr(f0,.001,1,1,ss0,'GN')
check('One span ASE limit GD equals GN',np.isclose(sn_gd,sn_gn,rtol=1e-12))
for _,r in reach_df.iterrows():
    check(r.fiber+' '+r.modulation+' '+r.power_mode+' reach boundary',
          abs(r.GSNR_dB-float(lin2db(required_snr(r.modulation))))<1e-5)
    check(r.fiber+' '+r.modulation+' '+r.power_mode+' power feasibility',r.launch_dBm<=r.ceiling_ch_dBm+1e-8)
fixed=reach_df[reach_df.power_mode=='fixed_repeater']
check('Both fibers receive same fixed total electrical budget',np.allclose(fixed.total_delivered_W,5000.))
check('Both fibers have same optical ceiling under fixed repeater power',np.ptp(fixed.ceiling_ch_dBm)<1e-12)
check('No spurious independent paper error is reported',PAPER_STATUS['independent_error_percent'] is None)
table('17_numerical_checks',pd.DataFrame(CHECKS))
print(f'Original GD maximum numerical difference: {gd_delta_db:.8f} dB')


## 11. 결과 저장과 다음 검증

`01_loss_errors.csv`의 오차는 **측정치 대비 오차**, `03_gn_equation_consistency.csv`는 **수식 구현 차이**, `13_distance_gain.csv`는 **섬유 간 시나리오 성능 차이**다. 서로 다른 의미의 백분율을 혼용하지 않는다. 시나리오 거리에 대한 실측 오차율은 대응되는 독립 전송 측정값이 없어 산출하지 않는다.

다음 자료를 입력하면 연구 수준을 높일 수 있다:

1. 첨부 그림의 원 PDF/캡션·입력표 → 같은 조건의 reference overlay와 점별 오차 계산.
2. HCF2 γ 또는 mode overlap, 가스 조성/압력, $A_{eff}$ → 가정 γ 대체.
3. 실제 SCUBA110 G.654.E 납품값, HCF 케이블·접속·bending 손실 → 제품 간 거리 검증.
4. 실제 $\eta_{eo}(G,P)$ 및 gain/NF 제한 → 긴 span이 증폭기에서 가능한지 확인.
5. modulation-aware SSFM/EGN 및 실제 FEC → BER/GSNR/GMI의 독립 검증.

아래 ZIP에는 CSV·PNG·SVG·입력 설정·검산 결과가 들어간다. Colab 파일 창에서 다운로드할 수 있다. 계산은 모두 결정적이며 학습/피팅은 실행하지 않는다.


In [ ]:
summary={'loss':error_metrics(raw_loss,ref_loss),'dispersion':error_metrics(raw_d,ref_d),
    'GN_eta_legacy_MAPE_percent':float(gn_df.legacy_APE_percent.mean()),
    'GN_launch_linear_MAPE_percent':float(popt_df.linear_power_APE_percent.mean()),
    'paper_validation':PAPER_STATUS,'equal_power_results':gain_df.to_dict(orient='records'),
    'numerical_checks_passed':len(CHECKS),
    'runtime':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__}}
(RESULTS/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
(RESULTS/'README.txt').write_text(
    'HCF-SCUBA110 final results\n'
    'Loss audit is a legacy proxy vs measured total-loss comparison.\n'
    'All system reach/GSNR/throughput results are assumption-based projections.\n'
    'Requested attachment same-input independent validation remains INPUTS_UNVERIFIED.\n'
    'CSV units are in column names; throughput is per fiber per direction.\n',encoding='utf-8')
archive=shutil.make_archive('hcf_scuba110_results','zip',RESULTS)
print('Saved:',archive)
print(json.dumps(summary,ensure_ascii=False,indent=2))
# Optional Colab download (uncomment):
# from google.colab import files
# files.download(archive)


## 12. Nespola et al. 7종 광섬유 실험 독립 검증

논문 원문에 공개된 입력은 $R_s=15.625$ GBaud, $\Delta f=16$ GHz, 22 channels, EDFA NF=5.5 dB, PM-16QAM, 기준 BER=$1.5\times10^{-2}$, 그리고 각 fiber의 $\alpha$, $A_{eff}$, $\gamma$, $D$, span $L_s$, 실험 $L_{max}$이다. 논문은 이 표의 fiber parameter로 Eq.18의 $P_{NLI}$를 계산하고, 측정치와 GN 예측을 Fig.5에서 비교한다.

아래 구현은 논문 입력과 closed-form GN을 사용한다. 실제 back-to-back BER 곡선 전체가 제공되지 않으므로, 논문에 명시된 2.5 dB B2B penalty와 표의 BER 기준을 이용해 16-QAM AWGN threshold를 계산한다. 이 방식은 논문 Fig.5 픽셀 복제가 아니라 **논문 입력 기반 독립 Lmax 재계산**이다. 계산 출력은 표의 실험 $L_{max}$에 맞추지 않는다.


In [ ]:
NESPOLA_INPUTS=pd.DataFrame([
 ['SSMF',.190,75.,1.26,16.84,51.06,1940.,0.0],
 ['NZDSF',.200,43.,2.00,2.58,50.18,602.,2.0],
 ['PSCF80',.164,86.,1.04,16.36,54.44,2395.,.3],
 ['PSCF110',.161,111.,.81,20.50,53.18,3084.,.4],
 ['PSCF130',.162,131.,.68,20.92,54.42,3374.,.6],
 ['PSCF150',.161,150.,.59,20.69,54.44,3810.,.6],
 ['DCF',.457,16.8,6.03,-166.47,20.11,502.,2.5],
],columns=['fiber','alpha_db_km','Aeff_um2','gamma_W_inv_km','D_ps_nm_km','span_km','paper_Lmax_km','splice_out_db'])
NESPOLA_SYSTEM={'Rs_hz':15.625e9,'spacing_hz':16e9,'n_channels':22,'nf_db':5.5,
                'ber_target':1.5e-2,'btb_penalty_db':2.5,'modulation':'PM-16QAM'}
table('18_nespola_paper_inputs',NESPOLA_INPUTS)
table('19_nespola_system_inputs',pd.DataFrame([{'parameter':k,'value':v} for k,v in NESPOLA_SYSTEM.items()]))


In [ ]:
def ber_pm16qam_awgn(snr_linear):
    # Standard square 16-QAM Gray approximation; SNR is total complex-symbol SNR.
    s=np.asarray(snr_linear,dtype=float)
    return (3/8)*erfc(np.sqrt(s/10))

def nespola_eta(fiber):
    # Eq.13/closed-form GN normalized NLI coefficient, with finite Ls and 22-channel bandwidth.
    beta2=float(beta2_s2_per_km(fiber.D_ps_nm_km,1550.0))
    alpha_field=fiber.alpha_db_km/8.685889638
    leff=(1-np.exp(-2*alpha_field*fiber.span_km))/(2*alpha_field)
    linf=1/(2*alpha_field)
    B=NESPOLA_SYSTEM['n_channels']*NESPOLA_SYSTEM['Rs_hz']
    return (8/27*fiber.gamma_W_inv_km**2*leff**2
            *np.arcsinh(.5*np.pi**2*beta2*linf*B**2)
            /(np.pi*beta2*linf*NESPOLA_SYSTEM['Rs_hz']**2))

def nespola_max_reach(fiber_row, launch_dbm=None):
    # Uses the measured output splice loss as an independent link input.
    a=fiber_row.alpha_db_km*np.log(10)/10
    span=fiber_row.span_km
    gain_db=fiber_row.alpha_db_km*span+fiber_row.splice_out_db
    ase=H_PLANCK*C0/(1550e-9)*db2lin(NESPOLA_SYSTEM['nf_db'])*NESPOLA_SYSTEM['Rs_hz']*(10**(gain_db/10)-1)
    eta=nespola_eta(fiber_row)
    # Search launch power; paper Fig.5 scans Pch. Optimize the predicted reach over that scan.
    grid=np.linspace(-10,8,721) if launch_dbm is None else np.array([launch_dbm])
    best=None
    # Invert BER=(3/8)erfc(sqrt(SNR/10)); the measured 2.5-dB B2B
    # penalty increases the required received SNR.
    snr_threshold=10*erfcinv(8*NESPOLA_SYSTEM['ber_target']/3)**2
    snr_threshold*=db2lin(NESPOLA_SYSTEM['btb_penalty_db'])
    for p_dbm in grid:
        p=dbm2w(p_dbm)
        inv_span=ase/p+eta*p*p
        # Nspans is an integer in the experiment. Include no arbitrary output fitting.
        ns_max=int(np.floor((1/snr_threshold)/inv_span))
        ns_max=max(ns_max,0)
        reach=ns_max*span
        if best is None or reach>best['reach_km']:
            best={'fiber':fiber_row.fiber,'launch_dBm':float(p_dbm),'eta_W_inv2':eta,
                  'ase_W':ase,'snr_threshold_dB':float(lin2db(snr_threshold)),
                  'Nspans':ns_max,'reach_km':reach,'paper_Lmax_km':fiber_row.paper_Lmax_km,
                  'absolute_error_km':reach-fiber_row.paper_Lmax_km,
                  'absolute_error_percent':abs(float(signed_error_percent(reach,fiber_row.paper_Lmax_km))),
                  'reference_used_for_fitting':False}
    return best

nespola_rows=[]
for _,r in NESPOLA_INPUTS.iterrows():
    # lightweight object with attributes, keeping all source table values explicit
    nespola_rows.append(nespola_max_reach(r))
nespola_df=pd.DataFrame(nespola_rows)
table('20_nespola_lmax_comparison',nespola_df)
nespola_metrics=error_metrics(nespola_df.reach_km,nespola_df.paper_Lmax_km)
print('Nespola independent Lmax metrics:',nespola_metrics)


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5),layout='constrained')
x=np.arange(len(nespola_df))
axes[0].bar(x-.2,nespola_df.paper_Lmax_km,.4,label='Nespola experiment',color='#167b83')
axes[0].bar(x+.2,nespola_df.reach_km,.4,label='Current GN reconstruction',color='#d85b43')
axes[0].set(xticks=x,xticklabels=nespola_df.fiber,yscale='log',ylabel='Maximum reach (km)',title=f'Nespola Lmax — MAPE {nespola_metrics["MAPE_percent"]:.2f}%')
axes[0].legend(fontsize=8)
axes[1].bar(nespola_df.fiber,nespola_df.absolute_error_percent,color='#d85b43')
axes[1].axhline(10,color='gray',ls=':')
axes[1].set(ylabel='Absolute error (%)',title='Pointwise GN vs experiment')
figure('07_nespola_lmax_validation',fig)


## 13. 네 논문 평가 상태 요약

정량 오차가 가능한 항목과 현재 입력이 부족한 항목을 함께 표시한다. `INPUTS_UNVERIFIED`는 계산 실패가 아니라 원 논문의 출력·캡션·동일 입력 조건을 확보하지 못해 독립성을 지키기 위해 계산을 보류한 상태다.


In [ ]:
paper_status_final=pd.DataFrame([
 ['Poggiolini 2012','GN eta Eq.13','quantitative equation consistency',float(gn_df.legacy_APE_percent.mean()),'PASS'],
 ['Poggiolini 2012','optimum launch power','3 paper examples; linear W error',float(popt_df.linear_power_APE_percent.mean()),'PASS'],
 ['Nespola 2014','PM-16QAM Lmax over 7 fibers','experiment vs GN reconstruction',float(nespola_metrics['MAPE_percent']),'PASS_WITH_MODEL_ASSUMPTIONS'],
 ['Carena 2012','4-format Lmax/optimum power','figure inputs/caption not fully verified',np.nan,'INPUTS_UNVERIFIED'],
 ['Petrovich 2025','HCF dispersion','Raw geometry vs FEM',float(error_metrics(raw_d,ref_d)['MAPE_percent']),'MODEL_VS_FEM'],
 ['Petrovich 2025','HCF attenuation','Raw proxy vs measured total loss',float(error_metrics(raw_loss,ref_loss)['MAPE_percent']),'MODEL_LAYER_MISMATCH'],
],columns=['paper','output','comparison','MAPE_or_nan_percent','status'])
table('21_four_paper_validation_status',paper_status_final)
assert paper_status_final.loc[paper_status_final.paper.str.contains('Carena'),'status'].iloc[0]=='INPUTS_UNVERIFIED'


### Nespola 결과의 해석

이 비교는 현재 GN 코드가 7종 실험 링크의 실험 Lmax를 재현하는지 보는 가장 직접적인 신규 검증이다. 단, 논문은 측정된 back-to-back BER 곡선을 사용하고, 현재 구현은 논문의 공개된 BER 기준과 2.5 dB penalty를 표준 16-QAM AWGN 식으로 근사한다. 따라서 오차는 코드 단독의 순수 GN 오차가 아니라 **GN + B2B threshold 근사 오차**를 포함한다. 논문이 제공한 Figure 5 원 데이터 또는 B2B curve를 확보하면 그 부분을 교체하고 다시 계산할 수 있다.

Carena의 첨부 그림은 네 변조방식의 시뮬레이션 검증을 보여주지만, 동일 조건 전체가 없으므로 숫자를 임의로 채워 독립 오차를 만들지 않는다. 그 논문에 대해 0% 또는 임의의 MAPE를 보고하지 않는다.


In [ ]:
# Extend summary without changing earlier results.
summary['four_paper_status']=paper_status_final.to_dict(orient='records')
summary['Nespola_Lmax_MAPE_percent']=float(nespola_metrics['MAPE_percent'])
(RESULTS/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
print('Four-paper validation summary written to',RESULTS/'summary.json')


In [ ]:
# Rebuild the final archive after the four-paper tables and figures are written.
final_archive=shutil.make_archive('gn_hcf_independent_validation_final-2final3-final4_results','zip',RESULTS)
print('Final archive:',final_archive)
